# Kathmandu Airbnb ML Analysis — Model 02: Price Prediction

**Task:** predict `base_price_npr` using property characteristics, amenities and stay timing.

**Data provenance:** the source price observations identify themselves as `Synthetic Sample Data`. This experiment is useful for demonstrating a reproducible ML workflow, but it is not evidence of live Kathmandu Airbnb pricing.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from train_price_model import prepare_data, train

DATA = Path("price_observations_clean.csv")
df = prepare_data(DATA)
print("Rows:", len(df))
print("Unique properties:", df.property_id.nunique())
display(df.head())


## 1. Target distribution and feature preparation

In [ ]:
display(df["base_price_npr"].describe())
display(df.groupby("property_type")["base_price_npr"].agg(["count", "mean", "median"]).sort_values("median", ascending=False))


## 2. Model comparison

We compare a median baseline, Linear Regression, Ridge Regression, Random Forest and Gradient Boosting. A group-based split holds out entire properties so that a property cannot appear in both train and test data.


In [ ]:
metrics, metadata = train(DATA, "model_output")
display(metrics)
print(metadata)


## 3. Evaluation interpretation

- MAE: average absolute error in NPR; lower is better.
- RMSE: penalizes larger errors more strongly; lower is better.
- R²: variance explained on the held-out set; closer to 1 is generally better.
- The model selection is based on RMSE and R² together, while reporting MAE transparently.
- The split is grouped by property_id to measure generalization to unseen properties.


In [ ]:
pred = pd.read_csv("model_output/heldout_test_predictions.csv")
display(pred.head(10))
print("Mean absolute error (NPR):", round(pred.absolute_error_npr.mean(), 2))


## 4. Limitations

The dataset has 1,000 synthetic price observations from 400 property IDs. A grouped test split is more conservative than a row-random split, but the sample is still small and synthetic. No claim of real market accuracy should be made. Before deployment, replace or supplement it with permitted, timestamped, verifiable price observations and evaluate on a later time period.
